In [16]:
import vrplib
import numpy as np
import math
import tempfile
import os
from scipy.spatial.distance import cdist
import pulp
import re
import glob
import random

# **Data**

In [17]:
def read_tsp_cappart_format(file_path):
    """
    Parses the TSP/TSPTW text files from the specified directory.
    Structure:
    - n (int)
    - n*n distance matrix entries
    - n*2 time window entries (ignored for TSP)
    - n x_coords (ignored)
    - n y_coords (ignored)
    """
    with open(file_path, 'r') as f:
        # split() handles all whitespace (newlines and spaces) automatically
        values = f.read().split()

    iterator = iter(values)
    
    try:
        # 1. Read Number of Nodes
        n = int(next(iterator))
        
        # 2. Read Distance Matrix (n x n)
        # The file contains a flattened list of integer distances
        c = []
        for i in range(n):
            row = []
            for j in range(n):
                val = float(next(iterator)) # Read as float first to be safe
                row.append(int(val))        # Convert to int as per your DIDP model type
            c.append(row)
            
        # The rest of the file (Time windows, coords) is ignored for pure TSP
        # but the iterator ensures we consumed exactly what we needed.
        num_locations = n
        travel_cost = c
        return num_locations, travel_cost

    except StopIteration:
        raise ValueError(f"File {file_path} ended unexpectedly.")

In [26]:
folder_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\n20"
# Get all .txt files
all_files = glob.glob(os.path.join(folder_path, "*.txt"))

# Select 20 random instances (or all if less than 20)
num_instances_to_test = 1
if len(all_files) > num_instances_to_test:
    selected_files = random.sample(all_files, num_instances_to_test)
else:
    selected_files = all_files

print(f"Found {len(all_files)} files. Selected {len(selected_files)} for testing.")
print("Selected Instances:")
for f in selected_files:
    print(f" - {os.path.basename(f)}")
print("-" * 50)
for i, file_path in enumerate(selected_files):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(selected_files)}] Processing: {instance_name}")
    try:
        # --- A. Read Data ---
        # Using the function you defined in previous cells
        num_locations, travel_cost = read_tsp_cappart_format(file_path)
    except Exception as e:
        print(f"   -> ERROR processing {instance_name}: {e}")

Found 100 files. Selected 1 for testing.
Selected Instances:
 - 0.txt
--------------------------------------------------

[1/1] Processing: 0.txt


# **TSP model**

In [7]:
def create_tsp_mtz_relaxed_model(num_locations, travel_cost):
    """
    Creates a TSP Linear Programming Relaxation model with MTZ subtour elimination.
    
    Args:
        n_nodes (int): Number of nodes (n)
        dist_matrix (list of lists): n x n cost matrix
        
    Returns:
        pulp.LpProblem: The defined model
    """
    
    # --- 1. Initialize Model ---
    mdl = pulp.LpProblem("TSP_MTZ_Relaxed", pulp.LpMinimize)

    # --- 2. Create Variables ---
    
    # x[i, j]: Flow variables (Continuous 0-1)
    # Represents fraction of travel from i to j
    x = {}
    for i in range(num_locations):
        for j in range(num_locations):
            if i != j:
                x[(i, j)] = pulp.LpVariable(f"x_{i}_{j}", 0, 1, pulp.LpContinuous)

    # u[i]: MTZ potential variables (Continuous)
    # Represents the order/sequence of node i in the tour
    # Defined only for nodes 1..n-1 (Node 0 is the start/anchor)
    # Bounds: 1 <= u_i <= n-1
    u = {}
    for i in range(1, num_locations):
        u[i] = pulp.LpVariable(f"u_{i}", 1, num_locations - 1, pulp.LpContinuous)

    # --- 3. Objective Function ---
    # Minimize sum(c_ij * x_ij)
    mdl += pulp.lpSum(travel_cost[i][j] * x[(i, j)] 
                      for i in range(num_locations)
                      for j in range(num_locations) if i != j), "Total_Cost"

    # --- 4. Constraints ---

    # (A) Assignment Constraints (Degree Constraints)
    for k in range(num_locations):
        # Outgoing flow = 1
        mdl += pulp.lpSum(x[(k, j)] for j in range(num_locations) if k != j) == 1, f"Out_{k}"
        
        # Incoming flow = 1
        mdl += pulp.lpSum(x[(i, k)] for i in range(num_locations) if i != k) == 1, f"In_{k}"

    # (B) MTZ Subtour Elimination Constraints
    # u_i - u_j + n * x_ij <= n - 1
    # Valid for all i, j in {1, ..., n-1}, i != j
    # This prevents cycles that do not include node 0
    for i in range(1, num_locations):
        for j in range(1, num_locations):
            if i != j:
                lhs = u[i] - u[j] + num_locations * x[(i, j)]
                rhs = num_locations - 1
                mdl += lhs <= rhs, f"MTZ_{i}_{j}"

    return mdl

# **Execution**

In [27]:
print(f"--- Building TSP Relaxation Model (n={num_locations}) ---")
mdl = create_tsp_mtz_relaxed_model(num_locations= num_locations, travel_cost= travel_cost)

# 2. Solve
print("--- Solving ---")
# Using standard solver
solver = pulp.PULP_CBC_CMD(msg=False) # Or pulp.CPLEX_CMD()
mdl.solve(solver)

# 3. Output
print(f"Status: {pulp.LpStatus[mdl.status]}")
print(f"LOWER BOUND (Objective): {pulp.value(mdl.objective)}")

# Check for fractional values (Classic in LP relaxation)
print("\nVariable Values:")
for v in mdl.variables():
    if v.varValue and v.varValue > 0.01:
        print(f"{v.name} = {v.varValue}")

--- Building TSP Relaxation Model (n=20) ---
--- Solving ---
Status: Optimal
LOWER BOUND (Objective): 331.59999999999997

Variable Values:
u_1 = 1.0
u_10 = 1.0
u_11 = 1.0
u_12 = 1.0
u_13 = 1.0
u_14 = 1.0
u_15 = 1.0
u_16 = 1.0
u_17 = 1.0
u_18 = 1.0
u_19 = 1.0
u_2 = 1.0
u_3 = 1.0
u_4 = 1.0
u_5 = 1.0
u_6 = 1.0
u_7 = 1.0
u_8 = 1.0
u_9 = 1.0
x_0_18 = 0.95
x_0_5 = 0.05
x_10_14 = 0.95
x_10_2 = 0.05
x_11_1 = 0.05
x_11_13 = 0.95
x_12_19 = 0.95
x_12_6 = 0.05
x_13_11 = 0.95
x_13_19 = 0.05
x_14_10 = 0.95
x_14_8 = 0.05
x_15_2 = 0.95
x_15_3 = 0.05
x_16_6 = 0.95
x_16_7 = 0.05
x_17_4 = 0.05
x_17_8 = 0.95
x_18_0 = 0.95
x_18_9 = 0.05
x_19_12 = 0.95
x_19_13 = 0.05
x_1_11 = 0.05
x_1_3 = 0.95
x_2_10 = 0.05
x_2_15 = 0.95
x_3_1 = 0.95
x_3_15 = 0.05
x_4_17 = 0.05
x_4_7 = 0.95
x_5_0 = 0.05
x_5_9 = 0.95
x_6_12 = 0.05
x_6_16 = 0.95
x_7_16 = 0.05
x_7_4 = 0.95
x_8_14 = 0.05
x_8_17 = 0.95
x_9_18 = 0.05
x_9_5 = 0.95
